In [ ]:
import argparse
import os
import sys
import time
import torch
import mlflow
import pandas as pd

from datasets import Dataset #load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    AutoProcessor
)
from functools import partial

In [ ]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT") #modify
USER = os.getenv("SLURM_JOB_USER") #modify

In [ ]:
input_model = "google/medgemma-1.5-4b-it"
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
model_output_name = f"{input_model}_finetuned"
json_file = f"/scratch/{SLURM_JOB_ACCOUNT}/data/structured_notes_gpt_oss_120b_srun.json"
batch_size = 2
cache_dir = f"/scratch/{SLURM_JOB_ACCOUNT}/hf-cache/hub"
max_tokens = 2048
num_workers = int(os.getenv("SLURM_CPUS_PER_TASK"))

output_model_dir = os.path.join(output_path, model_output_name)
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

In [ ]:
def preprocess(examples, tokenizer, max_tokens=2048):
    """Convert input_data/output pairs into tokenized chat format."""
    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for input_data, output in zip(examples["conversation"], examples["structured_note"]):

        # Build chat messages
        messages = [
            {
                "role": "system",
                "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
            {"role": "user", "content": input_data},
            {"role": "assistant", "content": output},
        ]

        # Apply chat template — tokenize full conversation
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        # Also build prompt-only part to know where assistant response starts
        prompt_messages = messages[:-1]
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # Tokenize full conversation
        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
        )

        # Tokenize prompt only to get its length
        prompt_tokenized = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_tokens,
            padding=False,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        prompt_len = len(prompt_tokenized["input_ids"])

        # Mask prompt tokens in labels — only compute loss on assistant response
        labels = [-100] * prompt_len + input_ids[prompt_len:]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    }

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(input_model, use_fast=True, cache_dir=cache_dir)
processor = AutoProcessor.from_pretrained(input_model, cache_dir=cache_dir)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    input_model,
    torch_dtype=torch.bfloat16,
    device_map=device,
    cache_dir=cache_dir
)

In [ ]:
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
## DATA
df = pd.read_json(json_file)[:2000]
dataset = Dataset.from_pandas(df, preserve_index=False)
split = dataset.train_test_split(test_size=0.05, seed=42)

raw_train = split["train"]
raw_val = split["test"]

print(f"  Train size: {len(raw_train)}")
print(f"  Val size:   {len(raw_val)}")

In [ ]:
preprocess_fn = partial(preprocess, tokenizer=tokenizer, max_tokens=max_tokens)

In [ ]:
tokenized_train = raw_train.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_train.column_names,
    num_proc=num_workers,
)

In [ ]:
tokenized_val = raw_val.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_val.column_names,
    num_proc=num_workers,
)

In [ ]:
training_args = TrainingArguments(
    disable_tqdm=False,
    output_dir=output_model_dir,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,
    load_best_model_at_end=True,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size*4,
    dataloader_num_workers=num_workers,
    ddp_find_unused_parameters=True,
    dataloader_pin_memory=True,
    save_safetensors=True,
    metric_for_best_model="eval_loss",
    eval_strategy="steps",
    eval_steps=300,
    num_train_epochs=1,
    report_to=["mlflow"],
    logging_steps=50,
    logging_strategy="steps",
    run_name=f"{model_output_name}_{os.environ.get('SLURM_JOB_ID', 'local')}",
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

In [ ]:
start_train = time.time()

trainer.train()

stop_train = time.time()

In [ ]:
elapsed = stop_train - start_train
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print(f"Training took: {hours}h {minutes}m {seconds}s")

In [ ]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    merged_output_dir,
    safe_serialization=False
)

tokenizer.save_pretrained(merged_output_dir)
processor.save_pretrained(merged_output_dir)

# Testing the finetuned model

In [ ]:
prompt = """Doctor: Hey, how are you doing today?

Patient: Hello doctor. I am feeling pain on the bottom right in my belly.

Doctor: How long has the pain been there?

Patient: It started yesterday evening and got worse during the night.

Doctor: Can you describe the pain? Is it sharp, dull, cramping, or something else?

Patient: It started as a dull ache, but now it feels sharp when I move or walk.

Doctor: On a scale from 1 to 10, how strong is the pain?

Patient: Around 7 out of 10.

Doctor: Have you noticed any nausea, vomiting, fever, or changes in appetite?

Patient: Yes, I feel nauseous and I did not want breakfast this morning. I also think I have a slight fever.

Doctor: Have you had diarrhea or constipation?

Patient: No diarrhea, but I have not gone to the bathroom since yesterday.

Doctor: Does anything make the pain better or worse?

Patient: Moving makes it worse. Lying still helps a little.

Doctor: Have you experienced this kind of pain before?

Patient: No, never this bad.

Doctor: Do you have any medical conditions or take any medications regularly?

Patient: No major medical conditions. I only take allergy medicine sometimes.

Doctor: Thank you. I would like to examine your abdomen now, especially the lower right side.

Patient: Okay.

Doctor: When I press here, does it hurt?

Patient: Yes, especially when you let go.

Doctor: I understand. Based on your symptoms and the examination, this could be appendicitis. I recommend blood tests and an abdominal scan as soon as possible.

Patient: Is it serious?

Doctor: It can become serious if untreated, but we caught it early. We will arrange further testing immediately.

Patient: Thank you, doctor.

Doctor: You're welcome. We will take good care of you."""

In [ ]:
def build_messages(example: str) -> list[dict]:
    return [
        {"role": "system",
        "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
        {"role": "user",   "content": example},
    ]

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
SAMPLING = dict(temperature=0.0, max_tokens=1000)

In [ ]:
llm = LLM(model=merged_output_dir, tensor_parallel_size=1, dtype="bfloat16")

In [ ]:
sampling_params = SamplingParams(**SAMPLING)

In [ ]:
outputs = llm.chat(build_messages(prompt), sampling_params, use_tqdm=True)

In [ ]:
print(outputs[0].outputs[0].text)